# Clase 210 — PySpark básico para datasets grandes

Requiere: `pip install pyspark==3.5.3`. Usaremos modo local con todos los cores.

In [ ]:
import os, tempfile, shutil
from pathlib import Path
WORK = Path(tempfile.gettempdir()) / 'spark_demo'
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(); os.chdir(WORK)

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import broadcast, col, rand, when

spark = (SparkSession.builder
         .master('local[*]')
         .appName('clase210')
         .config('spark.sql.shuffle.partitions', '8')   # bajo para dataset chico
         .config('spark.ui.port', '4042')
         .getOrCreate())
print('Spark version:', spark.version, '| cores:', spark.sparkContext.defaultParallelism)

## 1. Dataset sintético — taxi trips

In [ ]:
# 1M filas, 5 columnas
df = (spark.range(0, 1_000_000)
      .withColumn('zone_id', (rand(seed=1) * 50).cast('int'))
      .withColumn('fare', rand(seed=2) * 100 + 5)
      .withColumn('tip', rand(seed=3) * 20)
      .withColumn('date', F.expr("date_add(date '2024-01-01', cast(rand(seed=4) * 30 as int))")))
df.printSchema()
df.show(5)
print('rows:', df.count())

## 2. Lazy vs eager — observable en Spark UI

In [ ]:
import time

t0 = time.perf_counter()
df2 = df.filter(col('fare') > 30).select('zone_id', 'fare', 'tip')   # transformation = lazy
print(f'transformation: {(time.perf_counter() - t0) * 1000:.1f} ms (no ejecutó nada)')

t0 = time.perf_counter()
n = df2.count()   # action = ejecuta
print(f'action df2.count(): {time.perf_counter() - t0:.2f} s — devolvió {n:,} filas')

## 3. Broadcast join (tabla chica × tabla grande)

In [ ]:
zones = spark.createDataFrame(
    [(i, f'zone_{i}', 'Manhattan' if i < 20 else 'Brooklyn' if i < 40 else 'Queens') for i in range(50)],
    ['zone_id', 'name', 'borough'],
)

t0 = time.perf_counter()
joined = df.join(broadcast(zones), 'zone_id').groupBy('borough').agg(F.avg('fare').alias('avg_fare'), F.count('*').alias('n'))
joined.show()
print(f'broadcast join + agg: {time.perf_counter() - t0:.2f} s')

## 4. Particionado al escribir

In [ ]:
out_path = str(WORK / 'trips_partitioned')
df.write.mode('overwrite').partitionBy('date').parquet(out_path)

# Verificar estructura
subdirs = sorted([p.name for p in Path(out_path).iterdir() if p.is_dir()])[:5]
print('subdirs:', subdirs)

# Predicate pushdown: leer solo 1 partición
t0 = time.perf_counter()
one_day = spark.read.parquet(out_path).filter(col('date') == '2024-01-15').count()
print(f'lectura 1 día: {one_day:,} filas en {time.perf_counter() - t0:.2f} s (solo leyó esa partición)')

## 5. Skew + salting

In [ ]:
# Crear skew: 90% en zone_id=1
skewed = (spark.range(0, 1_000_000)
          .withColumn('zone_id', when(rand(seed=5) < 0.9, 1).otherwise((rand(seed=6) * 50).cast('int')))
          .withColumn('amount', rand(seed=7) * 100))

t0 = time.perf_counter()
skewed.groupBy('zone_id').agg(F.sum('amount').alias('total')).count()
print(f'groupBy SIN salting: {time.perf_counter() - t0:.2f} s')

# Salting: agregar columna random para repartir la key caliente
salted = skewed.withColumn('salt', (rand(seed=8) * 10).cast('int'))
t0 = time.perf_counter()
(salted.groupBy('zone_id', 'salt').agg(F.sum('amount').alias('partial'))
       .groupBy('zone_id').agg(F.sum('partial').alias('total')).count())
print(f'groupBy CON salting: {time.perf_counter() - t0:.2f} s')
print('(en datasets más grandes el speedup es 5-10×; acá es chico para terminar rápido)')

## 6. Spark SQL (mismo resultado, otra sintaxis)

In [ ]:
df.createOrReplaceTempView('trips')
zones.createOrReplaceTempView('zones')
spark.sql('''
  SELECT z.borough, AVG(t.fare) AS avg_fare, COUNT(*) AS n
  FROM trips t JOIN zones z USING (zone_id)
  GROUP BY z.borough
  ORDER BY avg_fare DESC
''').show()

In [ ]:
spark.stop()
print('Spark session detenida.')

## Ejercicio guiado

1. Descargá un mes de NYC Yellow Taxi parquet (~150 MB). Reemplazá el dataset sintético y midí tiempos.
2. Hacé `.explain('extended')` sobre un join — observá el plan físico de Catalyst.
3. Probá `df.cache()` antes de 3 agregaciones distintas; comparar vs sin cache.
4. Cambiá `spark.sql.shuffle.partitions` entre 4 y 200 para el mismo dataset — encontrá el sweet spot.
5. Migrá un script pandas existente a PySpark. Midí RAM peak (`memory_profiler`).

## Conclusiones

- Spark es overkill para <1 GB; vale la pena desde unos 10 GB en una sola máquina.
- Broadcast join + AQE matan el 80% de los problemas de performance.
- `partitionBy` al escribir es lo que hace que lecturas filtradas sean rápidas.
- Skew es el bug invisible; UI lo muestra, salting lo resuelve.